# **ДЗ 2. Нелинейные модели: важность признаков от дерева до бустингов**

### Что внутри

Ноутбук — практика к урокам **«Random Forest vs Decision Tree importances»**, **«Практические тонкости Feature importances: Catboost»** и **«Практические тонкости Feature importances: XGB и LightGBM»**. Пройдём по всей цепочке вопросов о важности признаков, восстановленной из структуры модели:

1. что именно даёт усреднение при переходе от дерева к лесу — и почему это про дисперсию, а не про смещение;
2. как impurity-важность систематически врёт в пользу «богатых» признаков, и как это увидеть руками;
3. сколько деревьев на самом деле нужно, чтобы важности перестали шататься;
4. почему у одной обученной модели XGBoost три разных ответа на вопрос «какой признак важный»;
5. четыре важности CatBoost и та единственная, которая умеет быть отрицательной;
6. что со всем этим делать на практике.

Опорная формула, вокруг которой построен первый блок — дисперсия среднего $M$ деревьев с попарной корреляцией $\rho$:

$$\operatorname{Var}\!\left(\frac{1}{M}\sum_{m=1}^{M} a_m\right) = \rho\,\sigma^2 + \frac{1-\rho}{M}\,\sigma^2.$$

### **Напоминание: что мы умеем на входе**

К этому моменту мы уже умеем собирать важность из структуры одного дерева:

___
1. **Impurity-важность** — сумма снижений критерия (Gini, энтропия, MSE) по всем разбиениям, где участвует признак, взвешенная числом объектов в узле.
2. **Важность леса** — та же величина, усреднённая по $M$ деревьям. Именно это лежит в `feature_importances_` у sklearn.
3. **Важность бустинга** — по построению уже не про «чистоту», а про снижение самой функции потерь.
___

И знаем два ограничения, которые никуда не денутся:

- impurity-важность считается **на обучающей выборке** — она про то, как признак помог подогнаться, а не про пользу на новых данных;
- она **смещена** в сторону признаков с большим числом возможных порогов разбиения.

Оба мы в этом ноутбуке увидим на числах.

Для начала — соберём всё, что понадобится.

In [ ]:
!pip install -q catboost


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import spearmanr

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool

Фиксируем случайность.

In [ ]:
RANDOM_STATE = 42

В качестве набора данных берём [Pima Indians Diabetes](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database) — тот же, на котором построены таблицы важностей в теории. Целевая переменная `Outcome` — есть ли у пациента диабет.

Подробный EDA здесь не нужен: данные уже числовые и без пропусков в привычном смысле.

In [ ]:
path = 'https://github.com/SadSabrina/open-xai-materials/raw/refs/heads/main/data/diabetes.csv'
data = pd.read_csv(path)

X = data.drop(columns='Outcome')
y = data['Outcome']

X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=RANDOM_STATE, test_size=0.25, stratify=y
)

labels = list(X_train.columns)
print('train:', X_train.shape, ' test:', X_test.shape)

## Блок 1. Дерево → лес: что именно даёт усреднение

В теории мы разложили ошибку на три части и показали, что усреднение $M$ деревьев бьёт по одному-единственному слагаемому — дисперсии. Проверим это руками.

Сначала — базовое сравнение: одно дерево против леса.

In [ ]:
tree = DecisionTreeClassifier(random_state=RANDOM_STATE)
tree.fit(X_train, y_train)

forest = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
forest.fit(X_train, y_train)

print('ROC-AUC одного дерева:', round(roc_auc_score(y_test, tree.predict_proba(X_test)[:, 1]), 3))
print('ROC-AUC леса        :', round(roc_auc_score(y_test, forest.predict_proba(X_test)[:, 1]), 3))

Теперь главное. Дисперсия в разложении — это разброс **от обучающей выборки к обучающей выборке**. Значит, чтобы её увидеть, надо переобучить модель много раз на разных выборках и посмотреть на разброс важностей.

Сделаем 30 бутстрап-повторов и на каждом обучим и дерево, и лес.

In [ ]:
def importances_over_resamples(make_model, n_repeats=30, seed=RANDOM_STATE):
    """Важности признаков, собранные по n_repeats бутстрап-выборкам."""
    rng = np.random.RandomState(seed)
    rows = []
    for r in range(n_repeats):
        idx = rng.choice(len(X_train), size=len(X_train), replace=True)
        model = make_model(r)
        model.fit(X_train.iloc[idx], y_train.iloc[idx])
        rows.append(model.feature_importances_)
    return pd.DataFrame(rows, columns=labels)

imp_tree = importances_over_resamples(
    lambda r: DecisionTreeClassifier(random_state=RANDOM_STATE)
)
imp_forest = importances_over_resamples(
    lambda r: RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
)

spread = pd.DataFrame({
    'mean_tree': imp_tree.mean(),
    'std_tree': imp_tree.std(ddof=1),
    'mean_forest': imp_forest.mean(),
    'std_forest': imp_forest.std(ddof=1),
})
spread['std_ratio'] = spread['std_tree'] / spread['std_forest']
spread.sort_values('mean_forest', ascending=False).round(4)

**Q1.** Возьмите признак, самый важный по версии леса (`mean_forest`). Во сколько раз у него разброс важности `std_tree` больше, чем `std_forest`? Ответ округлите до десятых.



In [ ]:
# ваш код здесь


Посмотрите на верхние строки таблицы: у сильных признаков `mean_tree` и `mean_forest` близки, а `std_forest` в 2–3 раза меньше `std_tree`. Это ровно то, что обещала теория: **усреднение гасит дисперсию, а не смещение**. Порядок признаков и сами средние держатся, шатание вокруг них падает.

(У слабых признаков средние разъезжаются заметнее — но это уже не про усреднение, а про то, что случайные подмножества признаков дают им шанс попасть в разбиение, до которого в одиночном жадном дереве очередь не доходила.)

**Q2.** Что изменилось при переходе от одного дерева к лесу?

`Выберите все верные утверждения в тренажере`

Теперь проверим саму формулу. Дисперсия среднего:

$$\operatorname{Var}\!\left(\frac{1}{M}\sum_m a_m\right) = \rho\,\sigma^2 + \frac{1-\rho}{M}\,\sigma^2,$$

где $\sigma^2$ — дисперсия прогноза одного дерева, а $\rho$ — попарная корреляция деревьев. Достанем и то, и другое прямо из обученного леса: у каждого дерева внутри `forest.estimators_` есть свой прогноз.

In [ ]:
def tree_predictions(fit_forest, X):
    """Матрица (число деревьев x число объектов) с прогнозами каждого дерева."""
    Xv = X.values if hasattr(X, 'values') else X   # деревья внутри леса обучены на массиве
    return np.vstack([t.predict_proba(Xv)[:, 1] for t in fit_forest.estimators_])

def sigma2_and_rho(fit_forest, X):
    P = tree_predictions(fit_forest, X)
    sigma2 = P.var(axis=0, ddof=1).mean()          # разброс деревьев, усреднённый по объектам
    corr = np.corrcoef(P)                          # попарные корреляции деревьев
    off = corr[np.triu_indices_from(corr, k=1)]
    return sigma2, np.nanmean(off)

sigma2, rho = sigma2_and_rho(forest, X_test)
print('sigma^2 =', round(sigma2, 4))
print('rho     =', round(rho, 4))

**Q3.** Чему равна попарная корреляция деревьев $\rho$ в лесу `forest`? Ответ округлите до сотых.

In [ ]:
# ваш код здесь


Подставим в формулу и посмотрим, во сколько раз усреднение сбило дисперсию — и куда она уйдёт в пределе.

In [ ]:
M = forest.n_estimators

var_mean = rho * sigma2 + (1 - rho) / M * sigma2
var_limit = rho * sigma2                     # предел при M -> бесконечность

print(f'дисперсия одного дерева : {sigma2:.4f}')
print(f'дисперсия среднего (M={M}): {var_mean:.4f}   -> в {sigma2/var_mean:.1f} раз меньше')
print(f'предел при M -> inf      : {var_limit:.4f}   -> в {1/rho:.1f} раз меньше, дальше падать некуда')

Вот отсюда и растёт вся конструкция случайного леса. Второе слагаемое $\frac{1-\rho}{M}\sigma^2$ гасится числом деревьев — это бесплатно, просто добавляй деревья. А первое, $\rho\sigma^2$, числом деревьев не убрать вообще: сколько деревьев ни добавляй, ниже него дисперсия не упадёт.

Единственный способ пробить этот пол — уменьшить саму $\rho$. Ровно для этого лес в каждом узле выбирает разбиение не по всем признакам, а по случайному подмножеству. Проверим, что это работает: обучим лес, которому разрешено смотреть на все признаки сразу.

In [ ]:
forest_all = RandomForestClassifier(
    n_estimators=100, max_features=None, random_state=RANDOM_STATE
)
forest_all.fit(X_train, y_train)

sigma2_all, rho_all = sigma2_and_rho(forest_all, X_test)

pd.DataFrame({
    'max_features': ['sqrt (по умолчанию)', 'None (все признаки)'],
    'rho': [round(rho, 3), round(rho_all, 3)],
    'предел дисперсии rho*sigma^2': [round(rho * sigma2, 4), round(rho_all * sigma2_all, 4)],
})

**Q4.** Чему равна $\rho$ у леса с `max_features=None`? Ответ округлите до сотых.

Сравните с Q3 и объясните себе знак разницы: деревья, которые в каждом узле перебирают одни и те же признаки, приходят к похожим разбиениям — а значит, и к похожим прогнозам.

In [ ]:
# ваш код здесь


## Блок 2. Смещение impurity-важности

В теории мы разобрали три источника смещения. Самый заметный — **перекос к «богатым» признакам**: чем больше у признака возможных порогов разбиения, тем выше шанс, что какой-то порог случайно даст большое снижение impurity.

Проверим это в чистом виде. Добавим к данным три признака, которые **не связаны с целевой переменной вообще никак**, и отличаются только числом возможных разбиений:

- `random_num` — непрерывный, порогов почти столько же, сколько объектов;
- `random_cat_50` — категориальный на 50 уровней;
- `random_cat_2` — бинарный, один-единственный возможный порог.

Если бы важность была честной, все три получили бы примерно ноль.

In [ ]:
rng = np.random.RandomState(RANDOM_STATE)

def add_dummies(df, rng):
    out = df.copy()
    out['random_num'] = rng.normal(size=len(df))
    out['random_cat_50'] = rng.randint(0, 50, size=len(df))
    out['random_cat_2'] = rng.randint(0, 2, size=len(df))
    return out

X_train_d = add_dummies(X_train, rng)
X_test_d = add_dummies(X_test, rng)

labels_d = list(X_train_d.columns)
X_train_d.head()

In [ ]:
forest_d = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
forest_d.fit(X_train_d, y_train)

imp_d = pd.Series(forest_d.feature_importances_, index=labels_d).sort_values(ascending=False)

plt.figure(figsize=(9, 4))
colors = ['#eb6834' if f.startswith('random_') else '#2a78d6' for f in imp_d.index]
bar = plt.barh(imp_d.index[::-1], imp_d.values[::-1], color=colors[::-1])
plt.title('Impurity-важность: оранжевым — заведомо бессмысленные признаки', pad=12)
plt.tight_layout();

In [ ]:
imp_d.round(4).to_frame('impurity_importance')

**Q5.** Какой из трёх фиктивных признаков получил наибольшую impurity-важность?

`Выберите один вариант в тренажере`

**Q6.** Какое место занимает `random_num` в общем рейтинге важностей (1 — самый важный)? Ответ — целое число.

In [ ]:
# ваш код здесь


Разница между тремя фиктивными признаками — не случайность и не «повезло». Она ровно та, что предсказывает теория: чем больше у признака кандидатов на разбиение, тем выше максимум Gini-gain **даже при полном отсутствии сигнала**. Это эффект максимально выбранной статистики, по природе тот же, что множественное тестирование: проверь достаточно порогов — какой-нибудь выстрелит случайно.

Посчитаем кандидатов явно. Важная оговорка: sklearn не умеет настоящих категориальных разбиений — целочисленный `random_cat_50` он видит как обычное число и режет порогом. Поэтому для всех трёх признаков число кандидатов — это просто число различных значений минус один.

**Q7.** Почему `random_num` обошёл `random_cat_2`, хотя оба одинаково бессмысленны?

`Выберите все верные утверждения в тренажере`

In [ ]:
n_thresholds = pd.DataFrame({
    'различных значений': [X_train_d[c].nunique() for c in ['random_num', 'random_cat_50', 'random_cat_2']],
    'impurity-важность': [round(imp_d[c], 4) for c in ['random_num', 'random_cat_50', 'random_cat_2']],
}, index=['random_num', 'random_cat_50', 'random_cat_2'])
n_thresholds['кандидатов на разбиение'] = n_thresholds['различных значений'] - 1
n_thresholds[['различных значений', 'кандидатов на разбиение', 'impurity-важность']]

Теперь второй источник смещения — **оценка на обучающей выборке**. Impurity-важность собрана из того, как признак помог подогнаться под `X_train`. Зададим другой вопрос: а насколько модель просядет на **тесте**, если этот признак испортить?

Это и есть permutation importance — первый post-hoc метод, который нам встретится. Здесь он нужен как контроль.

In [ ]:
perm = permutation_importance(
    forest_d, X_test_d, y_test,
    n_repeats=30, random_state=RANDOM_STATE, scoring='roc_auc'
)

compare = pd.DataFrame({
    'impurity (на train)': forest_d.feature_importances_,
    'permutation (на test)': perm.importances_mean,
}, index=labels_d).sort_values('impurity (на train)', ascending=False)

compare.round(4)

**Q8.** По impurity-важности `random_num` занимал 5 место (Q6). Какое место он занимает по permutation importance на тесте? Ответ — целое число.

Обратите внимание и на знак: у бессмысленного признака permutation importance уходит в минус — испортив его, модель на тесте становится чуть **лучше**.

In [ ]:
# ваш код здесь


## Блок 3. Сколько деревьев достаточно

Мы знаем, что разброс важностей падает как $\frac{1-\rho}{M}$. Значит, у добавления деревьев есть точка, после которой смысла почти нет: второе слагаемое уже мало по сравнению с полом $\rho\sigma^2$.

Посмотрим на это напрямую — как стабильность важностей зависит от числа деревьев.

In [ ]:
def importance_std_for_M(M, n_repeats=15, seed=RANDOM_STATE):
    rng = np.random.RandomState(seed)
    rows = []
    for r in range(n_repeats):
        idx = rng.choice(len(X_train), size=len(X_train), replace=True)
        f = RandomForestClassifier(n_estimators=M, random_state=RANDOM_STATE, n_jobs=-1)
        f.fit(X_train.iloc[idx], y_train.iloc[idx])
        rows.append(f.feature_importances_)
    return pd.DataFrame(rows, columns=labels).std(ddof=1).mean()

M_grid = [1, 2, 5, 10, 25, 50, 100, 200, 400]
stds = [importance_std_for_M(M) for M in M_grid]

stability = pd.DataFrame({'M': M_grid, 'средний std важностей': np.round(stds, 4)})
stability

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(M_grid, stds, marker='o', color='#2a78d6')
plt.xscale('log')
plt.xlabel('число деревьев M (лог. шкала)')
plt.ylabel('средний разброс важностей')
plt.title('Разброс важностей выходит на плато — дальше деревья почти не помогают', pad=12)
plt.grid(alpha=0.3)
plt.tight_layout();

**Q9.** Во сколько раз средний разброс важностей при $M=100$ меньше, чем при $M=1$? Ответ округлите до десятых.

In [ ]:
# ваш код здесь


А теперь посмотрите на правый край таблицы. Между $M=100$ и $M=400$ разброс почти не меняется — кривая вышла на плато. Это и есть $\rho\sigma^2$: слагаемое $\frac{1-\rho}{M}\sigma^2$ уже настолько мало, что новые деревья ничего не дают.

Практический смысл — ровно тот, что заложен в режим `async_mode` пакета `rfgboost`: перестать добавлять деревья, когда падение остановилось, и получить ту же точность меньшим числом деревьев.

## Блок 4. Бустинги: одна модель — три разных ответа

Переходим к бустингам. Здесь важность считается уже не по «чистоте», а по снижению функции потерь — и способов посчитать её сразу несколько. В XGBoost их три:

- **gain** — насколько разбиения по признаку улучшили целевую функцию;
- **cover** — сколько объектов (точнее, суммарный гессиан) эти разбиения задевают;
- **weight** — сколько раз признак вообще использовался.

Обучим одну модель и достанем из неё все три.

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.1,
    random_state=RANDOM_STATE, eval_metric='logloss'
)
xgb_model.fit(X_train, y_train)

booster = xgb_model.get_booster()

xgb_imp = pd.DataFrame({
    t: pd.Series(booster.get_score(importance_type=t))
    for t in ['gain', 'total_gain', 'cover', 'total_cover', 'weight']
}).reindex(labels)

xgb_imp.round(2)

Уже видно, что колонки ранжируют признаки по-разному. Померим это честно — ранговой корреляцией Спирмена.

In [ ]:
for a, b in [('gain', 'weight'), ('gain', 'cover'), ('cover', 'weight')]:
    r = spearmanr(xgb_imp[a], xgb_imp[b]).statistic
    print(f'Spearman({a:6s}, {b:6s}) = {r: .2f}')

**Q10.** Чему равна ранговая корреляция Спирмена между ранжированиями по `gain` и по `weight`? Ответ округлите до сотых.

In [ ]:
# ваш код здесь


**Q11.** Найдите признак, который поднимается по `weight` заметно выше, чем по `gain` (сравните места в двух ранжированиях). Что это означает?

`Выберите все верные утверждения в тренажере`

In [ ]:
ranks = pd.DataFrame({
    'место по gain': xgb_imp['gain'].rank(ascending=False).astype(int),
    'место по weight': xgb_imp['weight'].rank(ascending=False).astype(int),
})
ranks['сдвиг'] = ranks['место по gain'] - ranks['место по weight']
ranks.sort_values('сдвиг', ascending=False)

Проверим ещё одно утверждение из теории: `cover` — это `total_cover`, делённый на число разбиений по признаку, то есть на `weight`.

**Q12.** Возьмите признак `Glucose`. Чему равно `total_cover / weight`? Ответ округлите до целого и сравните с колонкой `cover`.

In [ ]:
# ваш код здесь


Теперь LightGBM. В теории мы сказали, что его `split` — это в точности `weight` из XGBoost по смыслу (число разбиений), а `gain` отличается способом агрегирования. Смысл тот же — числа разные.

In [ ]:
lgb_model = lgb.LGBMClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.1,
    random_state=RANDOM_STATE, verbose=-1
)
lgb_model.fit(X_train, y_train)

lgb_imp = pd.DataFrame({
    'split': lgb_model.booster_.feature_importance(importance_type='split'),
    'gain': lgb_model.booster_.feature_importance(importance_type='gain'),
}, index=labels)

both = pd.DataFrame({
    'XGB weight': xgb_imp['weight'],
    'LGBM split': lgb_imp['split'],
    'XGB total_gain': xgb_imp['total_gain'].round(1),
    'LGBM gain': lgb_imp['gain'].round(1),
})
both

**Q13.** Совпадают ли **числа** в колонках `XGB weight` и `LGBM split`? А ранжирования признаков по ним?

`Выберите верное утверждение в тренажере`

In [ ]:
print('числа совпадают     :', bool((both['XGB weight'] == both['LGBM split']).all()))
print('Spearman по рангам  :', round(spearmanr(both['XGB weight'], both['LGBM split']).statistic, 2))

## Блок 5. CatBoost: четыре важности и одна отрицательная

В CatBoost способов посчитать важность четыре. Нас интересуют два, которые видно на табличных данных:

- **PredictionValuesChange** — насколько в среднем меняется прогноз при изменении признака. Считается по умолчанию и нормирован так, что сумма по всем признакам равна 100.
- **LossFunctionChange** — насколько изменится **метрика**, если признак из модели убрать. В отличие от первой, эта важность знает, в какую сторону поехал прогноз — и поэтому **может быть отрицательной**.

In [ ]:
cb_model = CatBoostClassifier(
    iterations=200, depth=4, learning_rate=0.1,
    random_seed=RANDOM_STATE, verbose=0
)
cb_model.fit(X_train, y_train)

train_pool = Pool(X_train, y_train)

cb_imp = pd.DataFrame({
    'PredictionValuesChange': cb_model.get_feature_importance(type='PredictionValuesChange'),
    'LossFunctionChange (train)': cb_model.get_feature_importance(train_pool, type='LossFunctionChange'),
}, index=labels).sort_values('PredictionValuesChange', ascending=False)

cb_imp.round(4)

**Q14.** Чему равна сумма `PredictionValuesChange` по всем признакам? Ответ — целое число.

In [ ]:
# ваш код здесь


Обратите внимание: на обучающем Pool отрицательных значений нет вообще. Оно и понятно — на тех же данных, на которых модель училась, любой признак хоть немного, да помог подогнаться. Это ровно тот второй источник смещения, о котором говорил урок.

Чтобы важность начала отвечать на вопрос «а на новых данных этот признак вообще нужен?», её надо считать на **отложенной** выборке. Возьмём для этого данные с фиктивными признаками из Блока 2 — так сразу видно, кого метрика должна забраковать.

In [ ]:
cb_model_d = CatBoostClassifier(
    iterations=200, depth=4, learning_rate=0.1,
    random_seed=RANDOM_STATE, verbose=0
)
cb_model_d.fit(X_train_d, y_train)

lfc = pd.DataFrame({
    'на train': cb_model_d.get_feature_importance(Pool(X_train_d, y_train), type='LossFunctionChange'),
    'на test': cb_model_d.get_feature_importance(Pool(X_test_d, y_test), type='LossFunctionChange'),
}, index=labels_d).sort_values('на test')

print('отрицательных на train:', int((lfc['на train'] < 0).sum()))
print('отрицательных на test :', int((lfc['на test'] < 0).sum()))
lfc.round(4)

**Q15.** У какого признака `LossFunctionChange` на тестовом Pool оказалась самой отрицательной?

`Выберите один вариант в тренажере`

Отрицательное значение читается буквально: удаление признака **улучшило** метрику. Ни одна из важностей, которые мы считали до этого, такого сказать в принципе не умеет — они все измеряют «сколько признак поучаствовал», а не «стало ли от него лучше». А `random_num`, который по impurity-важности бодро сидел на 5 месте, здесь наконец получает то, что заслужил.

In [ ]:
# ваш код здесь


И последнее, что стоит удержать про CatBoost: четвёртая важность, **PredictionDiff**, устроена принципиально иначе — она считается **для пары объектов**, только для непрерывных признаков, и объясняет не модель целиком, а конкретное различие между двумя наблюдениями. По типу это **локальная** важность, в отличие от всех остальных в этом ноутбуке.

**Q16.** Сопоставьте метод и его тип.

`Выполните сопоставление в тренажере`

## Блок 6. Что со всем этим делать

Мы посчитали важность признаков восемью разными способами на одних и тех же данных. Соберём топ-3 у трёх семейств моделей и посмотрим, насколько они вообще согласны друг с другом.

In [ ]:
top3 = {
    'RandomForest': list(pd.Series(forest.feature_importances_, index=labels).nlargest(3).index),
    'XGBoost (gain)': list(xgb_imp['gain'].nlargest(3).index),
    'CatBoost (PVC)': list(cb_imp['PredictionValuesChange'].nlargest(3).index),
    'LightGBM (gain)': list(lgb_imp['gain'].nlargest(3).index),
}

for name, feats in top3.items():
    print(f'{name:17s}: {feats}')

common = set(top3['RandomForest']) & set(top3['XGBoost (gain)']) & set(top3['CatBoost (PVC)'])
print()
print('пересечение топ-3 у трёх семейств:', sorted(common))

**Q17.** Сколько признаков попали в топ-3 одновременно у RandomForest, XGBoost и CatBoost? Ответ — целое число.

In [ ]:
# ваш код здесь


**Q18.** Как наиболее корректно оценивать важность признаков?

`Выберите все верные утверждения в тренажере`

### Выводы

Пройдём по тому, что мы увидели своими глазами:

1. **Усреднение работает против дисперсии, а не против смещения.** Разброс важностей у леса упал в несколько раз по сравнению с одиночным деревом, а средние значения остались примерно теми же. Формула $\rho\sigma^2 + \frac{1-\rho}{M}\sigma^2$ показывает и предел: ниже $\rho\sigma^2$ дисперсия не упадёт никогда, сколько деревьев ни добавляй.

2. **Случайные подмножества признаков — это не украшение, а способ пробить этот предел.** Лес с `max_features=None` даёт заметно более коррелированные деревья, а значит, и более высокий пол дисперсии.

3. **Impurity-важность систематически врёт в пользу «богатых» признаков.** Три заведомо бессмысленных признака получили радикально разные важности, и порядок между ними задан числом возможных разбиений, а не связью с целевой переменной. Permutation importance на тесте ставит их на место.

4. **У одной обученной модели несколько разных ответов на вопрос «какой признак важный».** gain, cover и weight — это три разных вопроса, а не три способа ответить на один. Между фреймворками не совпадают даже одноимённые важности.

5. **Отрицательная важность — это не баг.** LossFunctionChange умеет сказать «без этого признака было бы лучше», потому что она единственная привязана к метрике, а не к факту участия в разбиениях.

Общий вывод тот же, к которому нас вёл весь блок: чем гибче модель, тем больше способов «восстановить» важность из её устройства — и тем меньше среди них согласия. Это и есть мотивация к post-hoc методам, с которых начнётся следующая часть курса: они задают вопрос снаружи модели и потому не зависят от того, как она устроена внутри.

На этом всё, друзья! Вы молодцы.

Встретимся в следующих домашних заданиях, \
Ваша команда курса : )